In [3]:
from AgentBasedModel import *
from AgentBasedModel.params_calibration.utils_calibration import *
from AgentBasedModel.params_calibration.calibrationv3.utils_calibration_v2 import *
import random
import numpy as np
import optuna

In [5]:
file_10s = "ethusdt_10s_300days.csv"
prices_10s = get_prices(file_10s)

SLICE = 1000
N_RUNS = 100
N_TICKS = 1000
target_params = pd.read_csv('mean_std_ethusdt_validation_data2.csv')
NORM_COEF_LR = 1 / target_params.loc[target_params["param"] == 'mean_on_ret2', "mean"].iloc[0]
num = ['mean_on_ret2', "std_on_ret2", "q1_on_ret2", "q5_on_ret2", "q95_on_ret2", "q99_on_ret2", "kurtosis_on_ret2", "skewness_on_ret", "autocorr_on_ret_1", "autocorr_on_ret_5", "autocorr_on_ret_10", "autocorr_on_abs_1", "autocorr_on_abs_5", "autocorr_on_abs_10"]
W_cov = pd.read_csv('weights(inv_covariance_matrix_from_real_data).csv').to_numpy()
W_mse = pd.read_csv('weights(inv only diag).csv').to_numpy()

In [6]:
def loss(t_params, m_params, w):
    num = ['mean_on_ret2', "std_on_ret2", "q1_on_ret2", "q5_on_ret2", "q95_on_ret2", "q99_on_ret2", "kurtosis_on_ret2", "skewness_on_ret", "autocorr_on_ret_1", "autocorr_on_ret_5", "autocorr_on_ret_10", "autocorr_on_abs_1", "autocorr_on_abs_5", "autocorr_on_abs_10"]
    errors = np.zeros((len(num), 1))
    id = -1
    for i in num:
        id += 1
        errors[id] = t_params.loc[t_params["param"] == i, "mean"].iloc[0] - m_params[i]
    ans = errors.T @ w @ errors
    # print(ans.shape)
    return float(ans[0][0])

In [7]:
def load_params(study, stor):
    cur_study = optuna.load_study(study_name=study,storage=stor)
    return cur_study.best_params

def get_moments(params):
    rows = []
    errors_full = []
    errors_diag = []
    for run_idx in range(N_RUNS):
        random.seed(100 + run_idx)
        np.random.seed(100 + run_idx)
        try:
            exchange = ExchangeAgent(volume=params['Exchange_Volume'], std=params['Std'], transaction_cost=params['Transaction_cost'], std_random=params['Std_Random'])
            simulator2 = Simulator(**{
                'exchange': exchange,
                'traders': [
                    *[Random(exchange, params['Exchange_Volume']) for _ in range(params['Random'])],
                    *[Fundamentalist(exchange, params['Exchange_Volume']) for _ in range(params['Fundamentalist'])],
                    *[Chartist(exchange, params['Exchange_Volume']) for _ in range(params['Chartist'])],
                    *[Universalist(exchange, params['Exchange_Volume']) for _ in range(params['Universalist'])],
                    *[MarketMaker(exchange, params['Exchange_Volume']) for _ in range(0)],
                ],
            }, avg_traders=params['Avg_Traders'], last_step=params['Last_Step'], last_ret=params['Last_Ret'], noisy_level=params['Noisy_Level'], norm_coef_lr=NORM_COEF_LR)

            simulator2.simulate(N_TICKS, silent=True)
            prices = np.array(simulator2.info.prices)
            model_params = pipeline(prices, is_print=0)
            rows.append(model_params)
            errors_full.append(loss(target_params, model_params, W_cov))
            errors_diag.append(loss(target_params, model_params, W_mse))
        except Exception as e:
            print(e)
            continue
    moments_df = pd.DataFrame(rows)
    ans = pd.DataFrame({"param": moments_df.mean(numeric_only=True).index, "mean": moments_df.mean(numeric_only=True).values, "std": moments_df.std(numeric_only=True, ddof=1).values})
    return ans, float(np.mean(errors_full)), float(np.mean(errors_diag))


params_full1 = load_params("calibration_msm(exchange_volume=1000)", "sqlite:///calibration_msm(exchange_volume=1000).db")
params_full1["Exchange_Volume"] = 1000
params_full2 = load_params("calibration_msm(diff vol)4", "sqlite:///calibration_msm(diff vol)4")
params_full2['MarketMaker'] = 0
params_diag1 = load_params("calibration_mse(exchange_volume=1000)2", "sqlite:///calibration_mse(exchange_volume=1000)2.db")
params_diag1["Exchange_Volume"] = 1000
params_diag2 = load_params("calibration_mse(diff vol)", "sqlite:///calibration_mse(diff vol)")
params_diag2['MarketMaker'] = 0
dictt = {"W_full_run_1" : params_full1, "W_full_run_2" : params_full2, "W_diag_run_1" : params_diag1, "W_diag_run_2" : params_diag2}
for key, val in dictt.items():
    cur_moments_df, er_full, er_diag = get_moments(val)
    cur_moments_df.to_csv(f"calculated_moments/{key}_moments_mean_std.csv", index=False)

In [8]:
# посчитать среднее по метрикам из реального распределения + матрицу ковариаций, чтобы получился хорошая функция loss (нормированная адекватно) из статьи WINKER
df = pd.DataFrame()
step = 0
for i in range(0, prices_10s.shape[0] - SLICE + 1, SLICE):
    step += 1
    if step % 1000 == 0:
        print(step)
    cur_start = i
    cur_slice = prices_10s[cur_start:cur_start+SLICE]
    cur_params = pipeline(cur_slice, 0)
    ind = len(df)
    df.loc[ind, "start"] = int(cur_start)
    df.loc[ind, "loss_with_full_cov_matrix"] = loss(target_params, cur_params, W_cov)
    df.loc[ind, "loss_with_diag_cov_matrix"] = loss(target_params, cur_params, W_mse)

df

1000
2000
3000


,start,loss_with_full_cov_matrix,loss_with_diag_cov_matrix
0,0.0,6.128463,4.480967
1,1000.0,9.248655,5.663583
2,2000.0,42.011589,36.235349
3,3000.0,5.893103,4.021820
4,4000.0,4.312220,4.071656
...,...,...,...
3157,3157000.0,4.218379,2.838989
3158,3158000.0,12.144429,8.319292
3159,3159000.0,2.972741,4.735437
3160,3160000.0,10.694941,8.901532


In [10]:
q95_cov = df["loss_with_full_cov_matrix"].quantile(0.95)
q99_cov = df["loss_with_full_cov_matrix"].quantile(0.99)
q95_mse = df["loss_with_diag_cov_matrix"].quantile(0.95)
q99_mse = df["loss_with_diag_cov_matrix"].quantile(0.99)
print(f"95% quantile of loss with full cov matrix on real data: {q95_cov:.4f}")
print(f"99% quantile of loss with full cov matrix on real data: {q99_cov:.4f}")
print(f"95% quantile of loss with diag cov matrix on real data: {q95_mse:.4f}")
print(f"99% quantile of loss with diag cov matrix on real data: {q99_mse:.4f}")

95% quantile of loss with full cov matrix on real data: 38.4691
99% quantile of loss with full cov matrix on real data: 109.7717
95% quantile of loss with diag cov matrix on real data: 44.7933
99% quantile of loss with diag cov matrix on real data: 122.7447


In [9]:
for key, val in dictt.items():
    _, loss_full, loss_diag = get_moments(val)
    print(key)
    print(f"Error with W_full: {loss_full}")
    print(f"Error with W_diag: {loss_diag}")
    q_full = np.mean(df["loss_with_full_cov_matrix"].to_numpy() <= loss_full)
    q_diag = np.mean(df["loss_with_diag_cov_matrix"].to_numpy() <= loss_diag)
    print(f"quantile of my sim with full matrix: {q_full*100:.2f}%")
    print(f"quantile of my sim with diag matrix: {q_diag*100:.2f}%")
    print("-----------------------------------------------")

W_full_run_1
Error with W_full: 18.295635605454567
Error with W_diag: 21.739081787872873
quantile of my sim with full matrix: 83.78%
quantile of my sim with diag matrix: 85.99%
-----------------------------------------------
W_full_run_2
Error with W_full: 17.649248269854528
Error with W_diag: 24.151124728276002
quantile of my sim with full matrix: 82.95%
quantile of my sim with diag matrix: 87.89%
-----------------------------------------------
W_diag_run_1
Error with W_full: 29.70914901302526
Error with W_diag: 14.512079785287979
quantile of my sim with full matrix: 92.50%
quantile of my sim with diag matrix: 78.18%
-----------------------------------------------
W_diag_run_2
Error with W_full: 26.444903603154657
Error with W_diag: 12.644567695567343
quantile of my sim with full matrix: 90.80%
quantile of my sim with diag matrix: 74.26%
-----------------------------------------------
